# NZZ ContextAI — Experiment Runner

In [33]:
import sys
sys.path.insert(0, '../src')

import json, glob, tempfile, os, io, uuid, importlib
from datetime import datetime

import chromadb
import pandas as pd
import mlflow
from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display

# ── Config neu einlesen (wichtig bei Änderungen ohne Kernel-Neustart) ──────────
import config, evaluate
importlib.reload(config)
importlib.reload(evaluate)

from config import (
    CHUNK_SIZE, CHUNK_OVERLAP, MIN_TEXT_LENGTH,
    EMBEDDING_MODEL, RERANKER_MODEL, USE_RERANKING,
    CHROMA_PATH, CHROMA_COLLECTION,
    EVAL_GROUND_TRUTH, EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL,
    MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT,
    NZZ_JSON_GLOB,
)
from evaluate import evaluate_retrieval

# Monate die indexiert werden — None = alle verfügbaren
MONTHS = ['2025_12']

RUN_NAME = datetime.now().strftime('%Y-%m-%d_%H-%M')

print('Aktuelle Konfiguration aus config.py:')
print(f'  Chunk Size:       {CHUNK_SIZE} Wörter')
print(f'  Chunk Overlap:    {CHUNK_OVERLAP} Wörter')
print(f'  Embedding-Modell: {EMBEDDING_MODEL}')
print(f'  Reranking:        {"an — " + RERANKER_MODEL if USE_RERANKING else "aus"}')
print(f'  Top-K Retrieval:  {EVAL_TOP_K_RETRIEVAL}  →  Top-K Final: {EVAL_TOP_K_FINAL}')
print(f'  Monate:           {MONTHS or "alle"}')
print(f'  MLflow Run:       {RUN_NAME}')

Aktuelle Konfiguration aus config.py:
  Chunk Size:       1000 Wörter
  Chunk Overlap:    250 Wörter
  Embedding-Modell: Qwen/Qwen3-Embedding-0.6B
  Reranking:        an — cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
  Top-K Retrieval:  20  →  Top-K Final: 5
  Monate:           ['2025_12']
  MLflow Run:       2026-04-13_16-41


## 1. ChromaDB zurücksetzen
Die bestehende Collection wird geleert damit alte Chunks das Ergebnis nicht verfälschen.

In [34]:
client = chromadb.PersistentClient(path=CHROMA_PATH)

existing = [c.name for c in client.list_collections()]
if CHROMA_COLLECTION in existing:
    client.delete_collection(CHROMA_COLLECTION)
    print(f'Collection "{CHROMA_COLLECTION}" gelöscht.')

collection = client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={'hnsw:space': 'cosine'},
)
print(f'Collection "{CHROMA_COLLECTION}" neu erstellt — bereit für {len(MONTHS) if MONTHS else "alle"} Monat(e).')

Collection "chunks" gelöscht.
Collection "chunks" neu erstellt — bereit für 1 Monat(e).


## 2. Daten laden, chunken & embedden

In [35]:
from preprocess import _load_nzz_json, preprocess

BASE_DIR = os.path.abspath('../')
RAW_DIR  = os.path.join(BASE_DIR, 'data', 'raw')

if MONTHS:
    patterns = [os.path.join(RAW_DIR, f'articles_{m}.json') for m in MONTHS]
    dfs = [_load_nzz_json(p) for p in patterns]
    raw_df = pd.concat(dfs, ignore_index=True)
else:
    raw_df = _load_nzz_json()

df = preprocess(raw_df)
print(f'Artikel geladen: {len(raw_df):,}  →  nach Preprocessing: {len(df):,}')

Artikel geladen: 1,763  →  nach Preprocessing: 1,746


In [36]:
def chunk_article(row):
    words      = row['body'].split()
    article_id = str(row.get('article_id') or
                     uuid.uuid5(uuid.NAMESPACE_DNS, row['title'] + row['body'][:50]))
    chunks, start, index = [], 0, 0
    while start < len(words):
        text = row['title'] + '. ' + ' '.join(words[start:start + CHUNK_SIZE])
        if len(text.strip()) > 20:
            chunks.append({
                'article_id':     article_id,
                'chunk_id':       f'{article_id}-{index}',
                'chunk_index':    index,
                'chunk_text':     text,
                'title':          row['title'],
                'category':       row.get('category', ''),
                'author':         row.get('author', ''),
                'published_date': row.get('published_date', ''),
            })
            index += 1
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

all_chunks = [c for _, row in df.iterrows() for c in chunk_article(row)]
chunks_df  = pd.DataFrame(all_chunks)

print(f'Chunks gesamt:         {len(chunks_df):,}')
print(f'Ø Chunks pro Artikel:  {len(chunks_df)/len(df):.1f}')
print(f'Ø Chunk-Länge:         {chunks_df["chunk_text"].str.len().mean():.0f} Zeichen')

Chunks gesamt:         3,140
Ø Chunks pro Artikel:  1.8
Ø Chunk-Länge:         4483 Zeichen


In [37]:
QUERY_INSTRUCTION = (
    'Instruct: Gegeben eine Suchanfrage auf Deutsch, finde relevante Nachrichtenartikel '
    'aus dem NZZ-Archiv, die die Frage beantworten.\nQuery: '
)

print(f'Lade Embedding-Modell: {EMBEDDING_MODEL}')
embed_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True)

print(f'Generiere Embeddings für {len(chunks_df):,} Chunks...')
embeddings = embed_model.encode(
    chunks_df['chunk_text'].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

meta_cols = ['article_id', 'chunk_index', 'title', 'category', 'author', 'published_date']
for start in range(0, len(chunks_df), 500):
    batch = chunks_df.iloc[start:start + 500]
    collection.upsert(
        ids        = batch['chunk_id'].tolist(),
        embeddings = embeddings[start:start + 500],
        documents  = batch['chunk_text'].tolist(),
        metadatas  = batch[meta_cols].to_dict(orient='records'),
    )

print(f'✓ {collection.count():,} Chunks in ChromaDB gespeichert.')

Lade Embedding-Modell: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Generiere Embeddings für 3,140 Chunks...


Batches:   0%|          | 0/99 [00:00<?, ?it/s]

✓ 3,140 Chunks in ChromaDB gespeichert.


## 3. Evaluation

In [38]:
reranker = CrossEncoder(RERANKER_MODEL) if USE_RERANKING else None
if reranker:
    print(f'Reranker: {RERANKER_MODEL}')

with open(EVAL_GROUND_TRUTH) as f:
    ground_truth = [json.loads(l) for l in f if l.strip()]
print(f'Ground-Truth Queries: {len(ground_truth)}')

eval_rows, all_metrics = [], []

for sample in ground_truth:
    query        = sample['query']
    relevant_ids = sample['relevant_article_ids']

    q_emb = embed_model.encode(
        QUERY_INSTRUCTION + query, normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=[q_emb],
        n_results=EVAL_TOP_K_RETRIEVAL,
        include=['documents', 'metadatas', 'distances'],
    )
    chunks = [{
        'chunk_text':       doc,
        'article_id':       meta['article_id'],
        'title':            meta['title'],
        'category':         meta.get('category', ''),
        'similarity_score': 1 - dist,
    } for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    )]

    if reranker:
        scores = reranker.predict([(query, c['chunk_text']) for c in chunks])
        for c, s in zip(chunks, scores):
            c['rerank_score'] = float(s)
        chunks = sorted(chunks, key=lambda x: x['rerank_score'], reverse=True)[:EVAL_TOP_K_FINAL]
    else:
        chunks = chunks[:EVAL_TOP_K_FINAL]

    metrics = evaluate_retrieval(chunks, relevant_ids, k=EVAL_TOP_K_FINAL)
    all_metrics.append(metrics)

    top = chunks[0] if chunks else {}
    eval_rows.append({
        'Query':       query,
        'Hit@1':       metrics['hit_at_1'],
        'Hit@3':       metrics['hit_at_3'],
        f'Hit@{EVAL_TOP_K_FINAL}': metrics[f'hit_at_{EVAL_TOP_K_FINAL}'],
        'MRR':         round(metrics['mrr'], 3),
        'Top-1 Titel': top.get('title', '')[:60],
        'Score':       round(top.get('similarity_score', 0), 3),
    })

print(f'Fertig.')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Ground-Truth Queries: 14
Fertig.


In [39]:
n   = len(all_metrics)
avg = {k: sum(d[k] for d in all_metrics) / n for k in all_metrics[0]}

print(f'Hit@1: {avg["hit_at_1"]:.0%}   '
      f'Hit@3: {avg["hit_at_3"]:.0%}   '
      f'Hit@{EVAL_TOP_K_FINAL}: {avg[f"hit_at_{EVAL_TOP_K_FINAL}"]:.0%}   '
      f'MRR: {avg["mrr"]:.3f}   '
      f'NDCG@{EVAL_TOP_K_FINAL}: {avg[f"ndcg_at_{EVAL_TOP_K_FINAL}"]:.3f}')

def style_hit(val):
    if val == 1.0: return 'background-color: #c6efce; color: #276221'
    return 'background-color: #ffc7ce; color: #9c0006'

result_df = pd.DataFrame(eval_rows)
hit_cols  = ['Hit@1', 'Hit@3', f'Hit@{EVAL_TOP_K_FINAL}']

display(
    result_df.style
    .map(style_hit, subset=hit_cols)
    .format({'MRR': '{:.3f}', 'Score': '{:.3f}'})
    .set_caption(
        f'Chunk: {CHUNK_SIZE}W / {CHUNK_OVERLAP}W Overlap  |  '
        f'Modell: {EMBEDDING_MODEL.split("/")[-1]}  |  '
        f'Reranking: {"an" if USE_RERANKING else "aus"}'
    )
    .set_properties(**{'text-align': 'left', 'font-size': '13px'})
)

Hit@1: 86%   Hit@3: 100%   Hit@5: 100%   MRR: 0.917   NDCG@5: 0.938


,Query,Hit@1,Hit@3,Hit@5,MRR,Top-1 Titel,Score
0,Was erwarten Schweizer Anlageprofis für die Börsenmärkte im Jahr 2026?,1.000000,1.000000,1.000000,1.000,«Wir werden in spätestens 18 Monaten wieder Negativzinsen ha,0.677
1,Werden in der Schweiz wieder Negativzinsen eingeführt?,1.000000,1.000000,1.000000,1.000,«Wir werden in spätestens 18 Monaten wieder Negativzinsen ha,0.626
2,Wie verhält sich Putins Kriegsführung im Vergleich zu Stalin?,1.000000,1.000000,1.000000,1.000,«Putin ist viel draufgängerischer als Stalin»,0.685
3,Was zeigen sowjetische Archivdokumente über die Denkweise russischer Führungspersönlichkeiten?,1.000000,1.000000,1.000000,1.000,«Putin ist viel draufgängerischer als Stalin»,0.660
4,Wie werden ukrainische Kinder in Russland umerzogen?,1.000000,1.000000,1.000000,1.000,Die stummen Opfer des Krieges: wie Russland ukrainische Kind,0.681
5,Russland verschleppt ukrainische Kinder und gibt sie zur Adoption frei,1.000000,1.000000,1.000000,1.000,Die stummen Opfer des Krieges: wie Russland ukrainische Kind,0.669
6,Ukrainischer Freiwilliger birgt tote russische Soldaten vom Schlachtfeld,1.000000,1.000000,1.000000,1.000,«Da liegt ein Fuss im Feld»: Ein Ukrainer riskiert sein Lebe,0.648
7,Menschlichkeit im Krieg: Umgang mit gefallenen Feinden im Donbass,1.000000,1.000000,1.000000,1.000,«Da liegt ein Fuss im Feld»: Ein Ukrainer riskiert sein Lebe,0.569
8,Wie hat sich die Schweiz für die Zurückweisung jüdischer Flüchtlinge im Zweiten Weltkrieg entschuldigt?,1.000000,1.000000,1.000000,1.000,"«Der Bundesrat täte gut daran, sich für die gegenüber den ve",0.679
9,Kaspar Villiger Entschuldigung Bundesrat Holocaust 1995,1.000000,1.000000,1.000000,1.000,"«Der Bundesrat täte gut daran, sich für die gegenüber den ve",0.625


## 4. In MLflow loggen

In [40]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params({
        'embedding_model':  EMBEDDING_MODEL,
        'reranker_model':   RERANKER_MODEL if USE_RERANKING else 'none',
        'use_reranking':    USE_RERANKING,
        'chunk_size':       CHUNK_SIZE,
        'chunk_overlap':    CHUNK_OVERLAP,
        'top_k_retrieval':  EVAL_TOP_K_RETRIEVAL,
        'top_k_final':      EVAL_TOP_K_FINAL,
        'months_indexed':   str(MONTHS) if MONTHS else 'all',
        'chunks_in_db':     collection.count(),
        'num_eval_queries': len(ground_truth),
    })
    mlflow.log_metrics(avg)

    buf = io.StringIO()
    result_df.to_csv(buf, index=False)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv',
                                     delete=False, encoding='utf-8') as f:
        f.write(buf.getvalue())
        tmp = f.name
    mlflow.log_artifact(tmp, artifact_path='eval')
    os.unlink(tmp)

print(f'✓ Run "{RUN_NAME}" in MLflow gespeichert → http://localhost:5000')

✓ Run "2026-04-13_16-41" in MLflow gespeichert → http://localhost:5000
